# AI-102 Exercise 04a — Create a generative AI app that uses your own data

**Based on:** [Microsoft Learn — mslearn-ai-studio / 04a-use-own-data](https://github.com/MicrosoftLearning/mslearn-ai-studio/blob/main/Instructions/Exercises/04a-use-own-data.md)

## What you'll learn

- How the **RAG (Retrieval Augmented Generation)** pattern works
- Why grounding data matters vs. pure prompt engineering
- How to use the **Azure OpenAI Responses API** with **file search** (vector store)
- How to build a Python client that chats with grounded data

**Estimated time:** ~45 minutes  
**Level:** 300 (intermediate)

---

> ⚠️ **Note:** Some technologies used here are in preview. You may see warnings or minor errors — that's normal.

## Structure of this notebook

| Part | What | Where |
|------|------|-------|
| 1 | Create Azure AI Foundry project + deploy model | Portal (browser) |
| 2 | Explore the model without RAG | Portal playground |
| 3 | Add grounding data in the playground | Portal playground |
| 4 | Build the RAG client app | **This notebook — Python** |

---
## Part 1 — Create a Microsoft Foundry project 🌐

> **Do this in your browser — no code needed yet.**

1. Open [https://ai.azure.com](https://ai.azure.com) and sign in with your Azure credentials.
2. If prompted, enable the **New Foundry** option in the toolbar.
3. Create a new project with a unique name. In **Advanced options**, set:
   - **Foundry resource**: use the default (usually `{project_name}-resource`)
   - **Subscription**: your Azure subscription
   - **Resource group**: create or select one
   - **Region**: pick any **AI Foundry recommended** region
4. Click **Create** and wait for the project to be ready.

### Deploy the model

5. On the project home page → **Start building** → **Browse models**.
6. Search for **`gpt-4.1`**, review the card, and deploy it with default settings.
7. The model will open in the playground when ready.

✅ **Checkpoint:** You have a Foundry project and a deployed `gpt-4.1` model.

---
## Part 2 — Explore the model without RAG 🧪

> **Still in the browser — model playground.**

Let's see what the model knows *without* any grounding data.

1. In the playground, make sure **gpt-4.1** is selected.
2. Enter this query in the chat:
   ```
   Where can I stay in New York?
   ```
   → The model will give a generic answer (no specific Margie's Travel info).

3. Now add a **System message**:
   ```
   You are a travel assistant that provides information on travel services available from Margie's Travel.
   ```

4. Ask again: `Where can I stay in New York?`
   → Slightly better tone, but still no real data — may even hallucinate.

5. Try: `What destinations does Margie's Travel offer?`

🔍 **Key insight:** Prompt engineering can guide *tone and behavior*, but it can't provide *factual grounding* without real data. That's what RAG fixes.

---
## Part 3 — Add grounding data in the playground 📄

> **Still in the browser — Foundry portal.**

1. In a **new browser tab**, download the brochures archive:
   **[brochures.zip](https://microsoftlearning.github.io/mslearn-ai-studio/data/brochures.zip)**  
   Extract it to a folder called `brochures`.

2. Back in the playground → **Tools** section → **Upload files**.
3. Upload all the PDF brochures from the folder.

   > 💡 If you don't see a **Tools** section, make sure you're in single-model view (not comparison mode).

4. Resubmit: `Where can I stay in New York?`
   → Now the answer is grounded in the brochure PDFs. 🎉

5. Try: `What destinations does Margie's Travel offer?`

✅ **Checkpoint:** The playground now answers based on real brochure data.

---

## Part 4 — Build the RAG client app (Python) 🐍

Now we build the same thing programmatically, using the **Responses API** with **file search**.

### Get your credentials from the portal

Before running the cells below:

1. In the Foundry portal, go to the **Home** page of your project.
2. Note the **Project API key** and **Azure OpenAI Endpoint**.

You'll paste them into the credentials cell below.

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q openai python-dotenv

In [ ]:
# ── Step 2: Credentials ───────────────────────────────────────────────────────
#
# OPTION A (simplest): paste your credentials directly here.
#   Just replace the placeholder strings below and run the cell.
#   ⚠️  Don't commit this notebook to a public repo while keys are filled in!
#
# OPTION B (recommended for reuse): store as Colab Secrets.
#   Colab → 🔑 (left sidebar, "Secrets") → add API_KEY and AZURE_OPENAI_ENDPOINT
#   Then uncomment the userdata block below.
#
# OPTION C: upload a .env file and use python-dotenv (same as the original lab).
#   Create a .env file locally, upload via Colab file browser, then uncomment
#   the load_dotenv() block below.

# ---------- OPTION A: direct paste ----------
API_KEY = "your_api_key_here"                       # ← paste your Project API key
AZURE_OPENAI_ENDPOINT = "your_endpoint_here"        # ← paste your Azure OpenAI endpoint
MODEL_DEPLOYMENT = "gpt-4.1"                        # ← rename if your deployment has a different name

# ---------- OPTION B: Colab Secrets ----------
# from google.colab import userdata
# API_KEY = userdata.get('API_KEY')
# AZURE_OPENAI_ENDPOINT = userdata.get('AZURE_OPENAI_ENDPOINT')
# MODEL_DEPLOYMENT = userdata.get('MODEL_DEPLOYMENT') or 'gpt-4.1'

# ---------- OPTION C: .env file ----------
# import os
# from dotenv import load_dotenv
# load_dotenv()  # expects a .env file in the current directory
# API_KEY = os.getenv('API_KEY')
# AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
# MODEL_DEPLOYMENT = os.getenv('MODEL_DEPLOYMENT', 'gpt-4.1')

print("Endpoint:", AZURE_OPENAI_ENDPOINT[:40] + "..." if AZURE_OPENAI_ENDPOINT and len(AZURE_OPENAI_ENDPOINT) > 40 else AZURE_OPENAI_ENDPOINT)
print("Model:", MODEL_DEPLOYMENT)
print("API key set:", bool(API_KEY and API_KEY != 'your_api_key_here'))

In [ ]:
# ── Step 3: Import libraries & initialize the OpenAI client ───────────────────
import os
import glob
import zipfile
import urllib.request
from openai import OpenAI

# Initialize the OpenAI client using the Azure OpenAI endpoint
openai_client = OpenAI(
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=API_KEY
)

print("✅ OpenAI client initialized")

In [ ]:
# ── Step 4: Download & extract the brochures ──────────────────────────────────
#
# In the original lab you did this manually (download zip, extract to folder).
# Here we do it programmatically — same result, no manual steps needed.

BROCHURES_URL = "https://microsoftlearning.github.io/mslearn-ai-studio/data/brochures.zip"
BROCHURES_ZIP = "brochures.zip"
BROCHURES_DIR = "brochures"

if not os.path.isdir(BROCHURES_DIR) or not glob.glob(f"{BROCHURES_DIR}/*.pdf"):
    print("Downloading brochures...")
    urllib.request.urlretrieve(BROCHURES_URL, BROCHURES_ZIP)
    print("Extracting...")
    with zipfile.ZipFile(BROCHURES_ZIP, "r") as z:
        z.extractall(".")
    os.remove(BROCHURES_ZIP)
    print("Done.")
else:
    print("Brochures already present — skipping download.")

pdfs = glob.glob(f"{BROCHURES_DIR}/**/*.pdf", recursive=True) + glob.glob(f"{BROCHURES_DIR}/*.pdf")
pdfs = list(set(pdfs))
print(f"Found {len(pdfs)} PDF brochure(s):")
for p in sorted(pdfs):
    print(f"  {p}")

In [ ]:
# ── Step 5: Create a vector store and upload the brochures ────────────────────
#
# A vector store is where Azure OpenAI stores your documents as embeddings
# so the file_search tool can retrieve relevant chunks at query time.

print("Creating vector store and uploading files...")

vector_store = openai_client.vector_stores.create(name="travel-brochures")

file_streams = [open(f, "rb") for f in pdfs]
if not file_streams:
    raise RuntimeError("No PDF files found — re-run Step 4.")

file_batch = openai_client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id,
    files=file_streams
)

for f in file_streams:
    f.close()

print(f"✅ Vector store ready — {file_batch.file_counts.completed} file(s) indexed")
print(f"   Vector store ID: {vector_store.id}")

In [ ]:
# ── Step 6: Chat with your grounded travel assistant ──────────────────────────
#
# This is the RAG loop. Each turn:
#   1. You enter a question
#   2. The Responses API searches the vector store (file_search tool)
#   3. Relevant brochure chunks are injected into the context
#   4. The model answers based on *your actual data* — not hallucinations
#
# The previous_response_id maintains multi-turn conversation context.
#
# Type  quit  to exit.

SYSTEM_INSTRUCTIONS = (
    "You are a travel assistant that provides information on travel services "
    "available from Margie's Travel. Only answer questions based on the provided "
    "travel brochure data."
)

last_response_id = None

print("🌍 Margie's Travel Assistant — RAG Demo")
print("Type your question below. Enter 'quit' to exit.\n")

while True:
    input_text = input("You: ")
    if input_text.strip().lower() == "quit":
        print("Goodbye! 👋")
        break
    if not input_text.strip():
        print("Please enter a question.")
        continue

    # Get a response using the Responses API with file_search
    response = openai_client.responses.create(
        model=MODEL_DEPLOYMENT,
        instructions=SYSTEM_INSTRUCTIONS,
        input=input_text,
        previous_response_id=last_response_id,
        tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }]
    )

    print(f"\nAssistant: {response.output_text}\n")
    last_response_id = response.id

In [ ]:
# ── Step 7 (optional): Clean up Azure resources ───────────────────────────────
#
# When you're done with the exercise, delete your Azure resources to avoid
# ongoing costs. You can do this via the Azure portal:
#
#   https://portal.azure.com
#   → Open the resource group you created
#   → Toolbar → "Delete resource group"
#   → Confirm by typing the resource group name
#
# Optionally, delete just the vector store from the API:

# Uncomment to delete the vector store created in this session:
# openai_client.vector_stores.delete(vector_store.id)
# print(f"Vector store {vector_store.id} deleted.")

---
## Summary — What happened here?

```
User question
     │
     ▼
Responses API  ──[file_search tool]──► Vector Store
     │                                  (brochure PDFs)
     │◄── relevant chunks ─────────────┘
     │
     ▼
gpt-4.1  (with grounded context)
     │
     ▼
Answer based on YOUR data 🎯
```

### Key concepts reviewed

| Concept | What it does |
|---------|-------------|
| **Vector store** | Stores documents as embeddings for semantic search |
| **file_search tool** | Retrieves relevant chunks at query time |
| **Responses API** | Stateful API with tool use + multi-turn support via `previous_response_id` |
| **RAG pattern** | Grounds model responses in real data, eliminating hallucinations |

### Credentials reminder

- 🔒 **Delete or clear the API key** from the credentials cell before sharing this notebook.
- For recurring use, switch to **Colab Secrets** (Option B in Step 2) — keys never appear in the notebook file.

---
*AI-102 Exercise 04a · Microsoft Foundry · RAG with file search*